In [93]:
import os
import boto3
import requests
import yaml

# aws s3 credentials
with open('credentials.yml', 'r') as f: 
    credentials = yaml.safe_load(f)
    aws_access_key_id = credentials.get('aws').get('aws_access_key_id')
    aws_secret_access_key = credentials.get('aws').get('aws_secret_access_key')

# S3 Location
S3_BUCKET = 'cm-aws-s3-data-source'
S3_FOLDER_PATH = 'organization/sales'


In [ ]:
# INCREMENTAL LOAD
# 1. Get: API's latest version & AWS S3's versions
# 2. Compare & logging whether the latest version is already ingested
# 3. Execute the ingestion if not already ingested

# Practice: define script into tasks / functions
# e.g., Func1: Get API Latest version
# Func2: Get AWS S3 versions and compare if API latest version is already ingested -> True/False
# Func3: Execute the API latest version ingestion if Func2 return False

In [94]:
# Create aws session & s3 client
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
# create s3 client to interact with AWS S3
s3 = session.client(service_name='s3')

SA Metro GTFS
- Loading Mode: Initial Full Load + Incremental Load
    - Initial Full Load: Load the previous 10 versions of GTFS
    - Incremental Load:
        - Get the latest version in `landing zone` & latest version available from API.
        - Execute load if the API's latest version has not been ingested.  
- Partitioned Destination: `<landing_bucket>/gtfs/<FEED VERSION>`

In [95]:
# FULL LOAD
# 1. read data from SA Metro GTFS feed
# 2. upload feed version into landing bucket
BASE_URL = 'http://gtfs.adelaidemetro.com.au/v1'
LATEST_VERSION_NUMBER = 'static/latest/version.txt'
LATEST_VERSION_FEED = 'static/latest/google_transit.zip'

DESTINATION_BUCKET = 'cm-aws-s3-destination'
DOWNLOAD_PATH = 'downloaded'
S3_UPLOAD_PATH = 'k1/ivy/gtfs'

In [96]:
def get_latest_version():
    url = f'{BASE_URL}/{LATEST_VERSION_NUMBER}'

    response = requests.get(url)
    response.raise_for_status()

    return response.text.strip()


In [ ]:
def download_gtfs_feed():
    url = f"{BASE_URL}/{LATEST_VERSION_FEED}"

    response = requests.get(url)
    response.raise_for_status()

    return response.content


def download_gtfs_feed_by_version(version):
    """
    Download a specific version of GTFS feed.
    """
    url = f"{BASE_URL}/static/{version}/google_transit.zip"
    
    response = requests.get(url)
    response.raise_for_status()
    
    return response.content


def save_all_previous_locally(num_versions=10):
    """
    Download and save the N versions BEFORE the latest version locally,
    then upload them to S3.
    Uses the save_locally and upload_to_s3 functions.
    """
    print(f"Starting FULL LOAD of {num_versions} previous versions (local + S3)")

    # Get the latest version
    latest_version = get_latest_version()
    latest_version_num = int(latest_version)
    print(f"Latest version: {latest_version_num}")

    # Calculate the range: 10 versions BEFORE the latest (not including latest)
    start_version = latest_version_num - num_versions
    end_version = latest_version_num - 1

    print(f"Processing versions {start_version} to {end_version}\n")

    success_count = 0
    failed_versions = []

    for version_num in range(start_version, end_version + 1):
        version_str = str(version_num)

        try:
            print(f"Processing version {version_str}...")

            # Download GTFS feed for this version
            gtfs_bytes = download_gtfs_feed_by_version(version_str)
            print(f"  Downloaded")

            # Save locally using save_locally function
            local_path = save_locally(version_str, gtfs_bytes)
            print(f"  Saved locally at: {local_path}")

            # Upload to S3
            upload_to_s3(version_str, gtfs_bytes)

            success_count += 1
            print(f"  Version {version_str} completed successfully\n")

        except Exception as e:
            print(f"  Error processing version {version_str}: {str(e)}\n")
            failed_versions.append(version_str)
            continue

    print("="*50)
    print(f"FULL LOAD completed (local + S3)")
    print(f"Successfully processed: {success_count}/{num_versions} versions")
    if failed_versions:
        print(f"Failed versions: {failed_versions}")
    print("="*50)


def initial_full_load(num_versions=10):
    """
    Download and upload the N versions BEFORE the latest version.
    This is for initial full load.
    """
    print(f"Starting INITIAL FULL LOAD of {num_versions} versions")

    # Get the latest version
    latest_version = get_latest_version()
    latest_version_num = int(latest_version)
    print(f"Latest version: {latest_version_num}")

    # Calculate the range: 10 versions BEFORE the latest (not including latest)
    start_version = latest_version_num - num_versions
    end_version = latest_version_num - 1

    print(f"Loading versions {start_version} to {end_version}\n")

    success_count = 0
    failed_versions = []

    for version_num in range(start_version, end_version + 1):
        version_str = str(version_num)

        try:
            print(f"Processing version {version_str}...")

            # Download GTFS feed for this version
            gtfs_bytes = download_gtfs_feed_by_version(version_str)
            print(f"  Downloaded")

            # Save locally
            local_path = save_locally(version_str, gtfs_bytes)
            print(f"  Saved locally at: {local_path}")

            # Upload to S3
            upload_to_s3(version_str, gtfs_bytes)

            success_count += 1
            print(f"  Version {version_str} completed successfully\n")

        except Exception as e:
            print(f"  Error processing version {version_str}: {str(e)}\n")
            failed_versions.append(version_str)
            continue

    print("="*50)
    print(f"INITIAL FULL LOAD completed")
    print(f"Successfully loaded: {success_count}/{num_versions} versions")
    if failed_versions:
        print(f"Failed versions: {failed_versions}")
    print("="*50)

In [98]:
def save_locally(version, gtfs_bytes):
    os.makedirs(DOWNLOAD_PATH, exist_ok=True)

    file_path = os.path.join(
        DOWNLOAD_PATH,
        f'google_transit_{version}.zip'
    )

    with open(file_path, 'wb') as f:
        f.write(gtfs_bytes)

    return file_path


In [99]:
def upload_to_s3(version, gtfs_bytes):
    """
    Upload GTFS zip to S3 only if this version hasn't been ingested yet.
    """
    # List existing versions in S3
    response = s3.list_objects_v2(
        Bucket=DESTINATION_BUCKET,
        Prefix=f"{S3_UPLOAD_PATH}/full_load/",
        Delimiter="/"
    )

    existing_versions = []
    if "CommonPrefixes" in response:
        for prefix in response["CommonPrefixes"]:
            folder_name = prefix["Prefix"]
            existing_version = folder_name.split("version=")[-1].strip("/")
            existing_versions.append(existing_version)

    # Check if this version already exists
    if version in existing_versions:
        print(f"Version {version} already ingested. Skipping upload.")
        return  
    # Upload
    s3_key = (
        f"{S3_UPLOAD_PATH}/"
        f"full_load/"
        f"version={version}/"
        f"google_transit.zip"
    )

    s3.put_object(
        Bucket=DESTINATION_BUCKET,
        Key=s3_key,
        Body=gtfs_bytes
    )

    print(f"Uploaded version {version} to s3://{DESTINATION_BUCKET}/{s3_key}")


In [ ]:
def main():
    print("Starting GTFS FULL LOAD")
    print("="*50)

    # Step 1: Initial load - 10 previous versions
    print("\n[Step 1] Loading 10 previous versions...")
    initial_load(num_versions=10)

    # Step 2: Load latest version
    print("\n[Step 2] Loading latest version...")
    version = get_latest_version()
    print(f"Latest GTFS version: {version}")

    gtfs_bytes = download_gtfs_feed()
    print("GTFS feed downloaded")

    local_path = save_locally(version, gtfs_bytes)
    print(f"Saved locally at: {local_path}")

    upload_to_s3(version, gtfs_bytes)

    print("="*50)
    print("FULL LOAD completed successfully")

In [101]:
if __name__ == "__main__":
    main()


Starting GTFS FULL LOAD
Latest GTFS version: 1614
GTFS feed downloaded
Saved locally at: downloaded\google_transit_1614.zip
Version 1614 already ingested. Skipping upload.
FULL LOAD completed successfully


In [ ]:
# Execute the initial full load of 10 versions (including latest)
if __name__ == "__main__":
    initial_full_load(num_versions=10)

In [ ]:
def initial_load(num_versions=10):
    """
    Download the previous 10 versions of GTFS feed, save locally, and upload to S3.
    """
    # Get the latest version number from API
    latest_version = int(get_latest_version())
    
    # Create download directory
    os.makedirs(DOWNLOAD_PATH, exist_ok=True)
    
    # Loop through 10 versions before latest
    for version in range(latest_version - num_versions, latest_version):
        version_feed_url = f"{BASE_URL}/static/{version}/google_transit.zip"
        print(f"Downloading version {version}...")

        response = requests.get(version_feed_url)
        response.raise_for_status()

        initial_file_path = f"{DOWNLOAD_PATH}/google_transit_{version}.zip"

        # Collect chunks into bytes for both local save and S3 upload
        gtfs_bytes = b""
        for chunk in response.iter_content(chunk_size=8192):
            gtfs_bytes += chunk

        # Save locally
        with open(initial_file_path, "wb") as f:
            f.write(gtfs_bytes)
        print(f"  Saved to: {initial_file_path}")
        
        # Upload to S3
        upload_to_s3(str(version), gtfs_bytes)

    print("Initial load completed!")

In [ ]:
# Run initial load - download 10 previous versions locally and upload to S3
initial_load(num_versions=10)